In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

In [4]:
def power_law(x, A, alpha):
    return A * x ** alpha

def power_law_offset(x, A, alpha, C):
    return power_law(x, A, alpha) + C

def exponential(x, A, k, C):
    return A ** (k*x) + C

def logarithmic(x, A, C):
    return A * np.log(x) + B


funcs = {'power_law':power_law,
         'power_law_offset':power_law_offset,
         'exponential':exponential,
         'logarithmic':logarithmic}

In [10]:
def fmt(val):
    if 0 < abs(val) < 0.01:
        base, exp = f"{val:.1e}".split("e")
        return rf"{base}\times 10^{{{int(exp)}}}"
    return f"{val:.1f}"


def format_fit(symbol, func_name, popt):
    sym = str(symbol).replace("$", "")

    match func_name:
        case "power_law":
            A, alpha = popt
            return rf"{fmt(A)}{sym}^{{{fmt(alpha)}}}"
        case "power_law_offset":
            A, alpha, C = popt
            sign = "+" if C >= 0 else "-"
            return rf"{fmt(A)}{sym}^{{{fmt(alpha)}}} {sign} {fmt(abs(C))}"
        case "exponential":
            A, K, C = popt
            sign = "+" if C >= 0 else "-"
            return rf"{fmt(A)}e^{{{fmt(K)}{sym}}} {sign} {fmt(abs(C))}"
        case "logarithmic":
            A, C = popt
            sign = "+" if C >= 0 else "-"
            return rf"{fmt(A)}\ln({sym}) {sign} {fmt(abs(C))}"
        case _:
            params_str = ", ".join([fmt(v) for v in popt])
            return rf"\mathrm{{{func_name}}}({sym}; {params_str})"

In [3]:
# minimise reduced chi^2 to get the best model for a fit
def find_model(x, y, y_err, verbose=False):
    results = []
    for name, func in funcs.items():
        try:
            popt, pcov = curve_fit(func, x, y, sigma=y_err, absolute_sigma=True)
            perr = np.sqrt(np.diag(pcov))

            residuals = y - func(x, *popt)
            chi2 = np.sum((residuals / y_err) ** 2)

            dof = len(y) - len(popt)
            chi2_red = chi2 / dof
            results.append({
                    "Model": name,
                    "func": func,
                    "res": (popt, pcov),
                    "popt": popt,
                    "perr": perr,
                    "chi2": chi2,
                    "dof": dof,
                    "chi2_red": chi2_red})
            if verbose: print(results[-1])
            
        except Exception as e: # log0 error and such
            continue

    results_df = pd.DataFrame(results).sort_values("chi2_red").reset_index(drop=True)
    best = results_df.iloc[0].to_dict()
    return best, results_df
